# 08 - RS-PPO against ArmoRM: deliberately circular upper-bound test

This notebook deliberately trains PPO against ArmoRM and then evaluates against ArmoRM again. This is circular and is not a proxy-validity experiment.

## What this notebook can and cannot show

**Invalid because of circularity - do not report:**

- No RQ2 proxy validity: no claim of the form "R is a valid proxy for reward".
- No Spearman gate statistic as proxy validation. If rho is computed, it is diagnostic only and is marked as `circular_do_not_report: true`.
- No generalization beyond ArmoRM.

**Valid despite circularity:**

- **Upper-bound setup:** PPO against r_i makes delta_j approximately a reward-gradient step -> Assumption 1 holds almost by construction -> this is the best-case regime for f(p,R). Notebook 08 prepares geometry, reward_matrix, Wall-A, and LMC; Notebook 09 runs the binding final merge endpoint.
- **R-minus:** Does PPO break the conflict-free geometry? (purely geometric, 0 reward queries)
- **Wall A / R2:** Does U_p(theta(lambda)) remain nonlinear on the quality axes? (property of ArmoRM x interpolation, not a training artifact)
- **LMC:** Are theta_SFT and the PPO specialists linearly connected? (RS Working Hyp. 1)

**Critical condition for the upper bound:** the horizon must stay short. If the reward curve plateaus, PPO has converged, `delta != eta * grad r`, and the best-case argument loses its basis. Therefore `reward_plateaued` is logged per axis and carried into the verdict.

## What is trained?

- Exactly **5 PPO-LoRA adapters** are trained - one per ArmoRM head. These LoRA deltas are the `delta_i` and therefore the geometry.
- `theta_SFT` is **not** trained when `USE_BASE_AS_THETA_SFT=True`: TinyLlama-1.1B-Chat is used directly as the shared snapshot.
- ArmoRM is frozen, evaluation-only, and is never trained.
- Value heads are PPO infrastructure. They are saved separately and do **not** enter `delta_i` or `R`.

A single binding run is preregistered. Accept the result in either direction; no look-and-retrain loop.

---

### Audit fixes relative to the first draft

| # | Finding | Fix |
|---|---|---|
| F1 | `CFG` mutations were lost across the `subprocess` boundary -> adapters landed in the wrong directory, 4-bit/batch size/steps were ineffective | Training runs **in process** (`rs_ppo.run_ppo(...)`); CLI flags are also available in the script |
| F2 | ArmoRM head indices 0..4 were hard-coded (= the QRM bug class) | Index is resolved **by name from `config.id2label`** and asserted |
| F3 | Batched reward scoring was unchecked; attention masks via `!= pad_id` masked real `<|eot_id|>` tokens | Masks are built from **true lengths**; padding side is resolved **empirically**; `batched == single` is proven, otherwise fallback to `batch_size=1` |
| F4 | `pad_token_id or eos_token_id` -> falsy bug when `pad_token_id == 0` | explicit `is None` check |
| F5 | Head sanity ran on **HelpSteer2 reference answers**, not on the distribution PPO produces | Sanity runs on **theta_SFT generations** (16-32 tokens, identical generation config) |
| F6 | No plateau check -> upper-bound argument was unsupported | `detect_reward_plateau()` per axis, included in the verdict |
| F7 | `floor_collapse_risk` came from the sign of R-minus instead of the LP | Floor collapse is computed directly from `max min_i (Rv)_i <= tol` |
| F8 | Reward collection / merge / LMC were not implemented | implemented here: `reward_matrix.npy` over B with exact `src.merge`; LMC curves. Binding `lambda*` merge + bootstrap endpoint moved to Notebook 09 |
| F9 | Dead config keys (`MAX_NEW_TOKENS`, `REPETITION_PENALTY`, ...) | removed or actually wired through |


## Setup

Repository, dependencies, seeds, config.


In [ ]:
%cd /content

import os, sys, json, math, random, shutil, subprocess, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

repo_path = Path('/content/master-thesis')
repo_url = 'https://github.com/NZhang137/master-thesis.git'
if (repo_path / '.git').is_dir():
    print('Repository exists; pulling latest changes.')
    subprocess.run(['git', '-C', str(repo_path), 'pull', '--ff-only'], check=False)
else:
    print('Repository missing; cloning from GitHub.')
    if repo_path.exists():
        shutil.rmtree(repo_path)
    subprocess.run(['git', 'clone', repo_url, str(repo_path)], check=True)

%cd /content/master-thesis

!pip install -q "transformers==4.40.0" "peft==0.10.0" "accelerate==0.29.3" "trl==0.8.6" bitsandbytes datasets scipy numpy pandas matplotlib safetensors

import numpy as np
import pandas as pd
import torch

CONFIG = {
    'SEED': 137,
    'OUTPUT_DIR': 'results/rs_ppo_armorm_circular',
    'OUTPUT_ZIP': 'rs_ppo_armorm_circular_outputs.zip',

    'BASE_MODEL': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    'ARMORM_MODEL': 'RLHFlow/ArmoRM-Llama3-8B-v0.1',
    'ATTRIBUTES': ['helpfulness', 'correctness', 'coherence', 'complexity', 'verbosity'],

    # --- circularity is a deliberate, documented decision ---
    'CIRCULAR_ARMORM_ACKNOWLEDGED': True,

    # --- theta_SFT source ---
    'USE_BASE_AS_THETA_SFT': True,      # RUN_SFT only needed when this is False

    # --- phase switches (expensive stages behind explicit flags) ---
    'RUN_HEAD_SANITY': True,
    'RUN_SFT': False,              # only needed when USE_BASE_AS_THETA_SFT=False
    'RUN_PPO': False,
    'RUN_GEOMETRY': False,
    'RUN_REWARD_COLLECTION': False,   # F8: builds reward_matrix.npy over B (expensive)
    'RUN_LMC': False,                 # F8: theta_SFT <-> theta_i interpolation curves

    # --- F5: head sanity runs on POLICY GENERATIONS, not dataset responses ---
    'HEAD_SANITY_SAMPLES': 200,
    'HEAD_SANITY_SPLIT': 'validation',
    'HEAD_SANITY_ON_GENERATIONS': True,
    'HEAD_STD_MIN': 1e-6,
    'HEAD_UNIQUE_MIN': 10,

    # --- reward collection over the search set B ---
    'REWARD_NUM_PROMPTS': 80,
    'REWARD_PROMPT_SPLIT': 'validation',
    'REWARD_PROMPT_OFFSET': 160,      # disjoint from the v5 confirmatory slice [80:160]
    'N_GEN_PER_PROMPT': 1,             # NB08 reward matrix; NB09 final merge uses 4

    # --- PPO (RS Table 1 defaults; NO tuning -- "training is secondary") ---
    'PPO_AXES': ['helpfulness', 'correctness', 'coherence', 'complexity', 'verbosity'],
    'PPO_BATCH_SIZE': 64,
    'PPO_MINI_BATCH_SIZE': 8,
    'PPO_TOTAL_STEPS': 200,
    'PPO_N_PROMPTS': 2005,
    'ARMORM_REWARD_BATCH_SIZE': 8,
    'ARMORM_LOAD_IN_4BIT': True,

    # --- F6: short-horizon guard ---
    'PLATEAU_WINDOW': 50,
    'PLATEAU_SLOPE_EPS': 0.02,

    # --- selection / evaluation ---
    'M1PLUS_RHO': 0.5,
    'C1PP_C': 0.5,                    # C1++ trust-region radius factor
    'C1PP_EPS': 1e-8,
    'DM_DENOM_MIN': 1e-3,             # guard: Delta m% denominator
    'BOOTSTRAP_N': 2000,
    'BOOTSTRAP_SEED': 137,
    'SEARCH_SET_SEED': 137,
    'SEARCH_SET_DIRICHLET': 64,
    'LMC_GRID': [0.0, 0.25, 0.5, 0.75, 1.0],
    'FLOOR_TOL': 1e-9,                # F7: floor collapse iff LP value <= tol
    'WALL_A_R2_THRESHOLD': 0.3,     # frozen before binding run; changing later = p-hacking
    'MIN_GPU_MEMORY_GB': 35.0,
}

PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = (PROJECT_ROOT / CONFIG['OUTPUT_DIR']).resolve()
RS_RUNS_DIR = OUTPUT_DIR / 'rs_runs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RS_RUNS_DIR.mkdir(parents=True, exist_ok=True)

random.seed(CONFIG['SEED']); np.random.seed(CONFIG['SEED']); torch.manual_seed(CONFIG['SEED'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['SEED'])

assert CONFIG['CIRCULAR_ARMORM_ACKNOWLEDGED'] is True
assert CONFIG['SEED'] == 137
assert CONFIG['REWARD_PROMPT_OFFSET'] >= 160, 'must not overlap the v5 confirmatory slice'

print('Project root:', PROJECT_ROOT)
print('Output dir:  ', OUTPUT_DIR)
print('RS runs dir: ', RS_RUNS_DIR)
print(json.dumps(CONFIG, indent=2, sort_keys=True))


## Import, Firewall Test, and Pre-Registration

**F1:** `train_rs_ppo` is imported and configured **in process**. There is no longer a `subprocess` boundary, so `out_dir`, `batch_size`, `total_ppo_steps`, `n_prompts`, and 4-bit settings now actually take effect. Previously, exactly these five assignments were lost across the process boundary; adapters would have been written to `./rs_runs/` instead of `OUTPUT_DIR/rs_runs/`, and the "Geometry" cell would not have found them.

The firewall must fail hard **without** the acknowledgement flag and emit a loud circularity warning **with** the flag. Both paths are tested.


In [ ]:
import importlib
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

rs_ppo = importlib.import_module('scripts.train_rs_ppo')
rs_ppo = importlib.reload(rs_ppo)
from src.helpsteer2_utils import HELPSTEER2_ATTRIBUTES
assert tuple(rs_ppo.ATTRIBUTES) == tuple(HELPSTEER2_ATTRIBUTES), (
    f"axis order drift: {rs_ppo.ATTRIBUTES} vs {HELPSTEER2_ATTRIBUTES}")
assert tuple(CONFIG['ATTRIBUTES']) == tuple(HELPSTEER2_ATTRIBUTES), (
    f"notebook axis order drift: {CONFIG['ATTRIBUTES']} vs {HELPSTEER2_ATTRIBUTES}")

# --- F1: in-process overrides. These now actually take effect. ---
rs_ppo.apply_overrides(
    out_dir=str(RS_RUNS_DIR),
    batch_size=CONFIG['PPO_BATCH_SIZE'],
    mini_batch_size=CONFIG['PPO_MINI_BATCH_SIZE'],
    total_ppo_steps=CONFIG['PPO_TOTAL_STEPS'],
    n_prompts=CONFIG['PPO_N_PROMPTS'],
    armorm_model=CONFIG['ARMORM_MODEL'],
    armorm_load_in_4bit=CONFIG['ARMORM_LOAD_IN_4BIT'],
    armorm_reward_batch_size=CONFIG['ARMORM_REWARD_BATCH_SIZE'],
    plateau_window=CONFIG['PLATEAU_WINDOW'],
    plateau_slope_eps=CONFIG['PLATEAU_SLOPE_EPS'],
)
assert rs_ppo.CFG['out_dir'] == str(RS_RUNS_DIR), 'override did not stick'
assert rs_ppo.CFG['total_ppo_steps'] == CONFIG['PPO_TOTAL_STEPS']
print('[cfg] rs_ppo.CFG out_dir =', rs_ppo.CFG['out_dir'])
print('[cfg] rs_ppo.CFG steps   =', rs_ppo.CFG['total_ppo_steps'],
      ' batch =', rs_ppo.CFG['batch_size'], ' 4bit =', rs_ppo.CFG['armorm_load_in_4bit'])

# --- firewall: must BLOCK without the flag ---
blocked = False
try:
    rs_ppo.check_reward_firewall('helpfulness', CONFIG['ARMORM_MODEL'],
                                 circular_armorm_acknowledged=False)
except AssertionError as error:
    blocked = True
    print('Firewall correctly blocks unacknowledged ArmoRM PPO:', error)
assert blocked, 'FIREWALL BROKEN: ArmoRM PPO was not blocked without acknowledgement.'

# --- firewall: must WARN LOUDLY with the flag ---
firewall_ack = rs_ppo.check_reward_firewall('helpfulness', CONFIG['ARMORM_MODEL'],
                                            circular_armorm_acknowledged=True)
assert firewall_ack['circularity_acknowledged'] is True
assert 'RQ2 (proxy validity)' in firewall_ack['retired_research_questions']
print('Acknowledged circularity warning:', firewall_ack['warning'])


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + '\n',
                    encoding='utf-8')
    assert path.exists(), f'Missing JSON output: {path}'


def write_numpy(path, values):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(path, np.asarray(values))
    assert path.exists(), f'Missing NumPy output: {path}'


PREREGISTRATION_PATH = OUTPUT_DIR / 'preregistration.json'
pre_registration = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'experiment': 'RS-faithful PPO against ArmoRM, deliberately circular upper-bound test',
    'seed': CONFIG['SEED'],
    'theta_sft_source': ('TinyLlama-1.1B-Chat used directly as theta_SFT (already '
                         'instruction-tuned); deliberate deviation from RS pretrain->SFT->PPO. '
                         'The only requirement is a SHARED init across all PPO runs, '
                         'which is satisfied.'),
    'circularity': {
        'acknowledged': True,
        'retired_research_questions': ['RQ2 (proxy validity)'],
        'rationale': ('PPO reward model and evaluation model are both ArmoRM. This retires '
                      'proxy-validity claims but defines an UPPER-BOUND test: short-horizon PPO '
                      'against r_i makes Assumption 1 (delta ~ eta*grad r) approximately true by '
                      'construction, so this is the best-case regime for f(p,R). If f(p,R) does '
                      'not beat lambda=p here, it beats it nowhere.'),
    },
    'primary': {
        'metric': ('Delta U_p = mean over the 80 held-out prompts of '
                   'p^T ( r(theta(lambda*_M1+), prompt) - r(theta(p), prompt) ) -- the PAIRED '
                   'per-prompt difference on the native ArmoRM reward scale (raw ArmoRM head values)'),
        'success': 'mean Delta U_p > 0 with bootstrap 95% CI (2000 resamples) excluding 0',
        'bootstrap_n': CONFIG['BOOTSTRAP_N'],
        'bootstrap_seed': CONFIG['BOOTSTRAP_SEED'],
        'binding_run': True,
        'real_merges_required': True,
        'why_not_rank_normalised': (
            'The metric must be PAIRED and per-prompt to be bootstrappable over the 80 prompts. '
            'Rank-normalized U_p is a SET-LEVEL quantity (ranks taken over the search set B) and '
            'cannot be resampled prompt-wise. It is reported as a secondary metric, '
            'together with a paired rank transform (ranks pooled over both models), which IS '
            'bootstrappable. Fixed BEFORE the binding run -- changing it afterwards = p-hacking.'),
    },
    'secondary_metrics': {
        'delta_U_p_rank': 'paired rank-normalized Delta U_p (ranks pooled over both models)',
        'delta_m_percent_gain': 'Delta m%(lambda*) - Delta m%(p); specialists = STL reference',
        'sign_test': 'one-sided sign test over preferences (v5 reference: 11/12, p=0.0032)',
        'portfolio_certificates': 'C1++/M1++/P2++/P3++ returning p = floor certificate',
    },
    'secondary_non_circular': {
        'R_minus_nonzero': 'yes/no',
        'floor_collapse': 'from LP: max_{1^T v = 0} min_i (Rv)_i <= tol',
        'wall_A_R2': 'per axis, linear vertex fit against true vertex rewards',
        'wall_A_R2_threshold': CONFIG['WALL_A_R2_THRESHOLD'],
        'wall_A_rule': 'Wall A stands iff max R2 over the quality axes < threshold.',
        'LMC': 'smooth yes/no along theta_SFT <-> theta_i',
    },
    'horizon_guard': {
        'requirement': 'reward curve must NOT plateau',
        'rationale': ('The upper-bound reading rests on delta ~ eta*grad r, i.e. a SHORT horizon. '
                      'If PPO converges, Assumption 1 breaks again and the best-case claim '
                      'loses its basis.'),
        'window': CONFIG['PLATEAU_WINDOW'],
        'eps': CONFIG['PLATEAU_SLOPE_EPS'],
    },
    'negative_control': 'quality-heavy preferences MUST run (Wall A predicts failure there)',
    'interpretation_rule': {
        'quality_success': 'Wall A was not a hard cap; strong positive result',
        'quality_failure': ('upper-bound negative result: endpoint-linear rules provably cannot '
                            'trace a nonlinear landscape, even in the best-case regime'),
    },
    'valid_claims': ['upper bound', 'R-minus', 'wall A (R2)', 'LMC'],
    'invalid_claims': ['proxy validity', 'generalization beyond this reward model'],
    'config': CONFIG,
}
write_json(PREREGISTRATION_PATH, pre_registration)
print('Wrote:', PREREGISTRATION_PATH)


## Phase 0 - VRAM, ArmoRM Head Verification, Batching Proof, Head Sanity

Before head sanity, a single `theta_SFT` path is prepared. If `USE_BASE_AS_THETA_SFT=True`, TinyLlama-1.1B-Chat is saved there as a snapshot; otherwise, a dedicated SFT phase must create that path.

Four checks, each a direct lesson from the QRM issue:

1. **VRAM** - ArmoRM 8B (4-bit) + TinyLlama policy + value head + reference model must fit at the same time. Below about 35 GB: hard abort.
2. **F2 - Head index by name.** `config.id2label` is read and each of the five heads is resolved **by name**. No hard-coded index. This exact line would have caught `verbosity == 0` in QRM within one second.
3. **F3 - Batching proof.** `batched == single` is proven and the padding side is resolved empirically. If either check fails -> fallback to `batch_size=1`.
4. **F5 - Sanity on the correct distribution.** The heads are checked on **theta_SFT generations** (16-32 tokens, identical generation config to PPO), *not* on HelpSteer2 reference answers. A head can vary nicely on human text and collapse on short TinyLlama rollouts.

If an assertion fails: **do not train.**


In [ ]:
SFT_MERGED = RS_RUNS_DIR / 'theta_sft' / 'merged'

if CONFIG['USE_BASE_AS_THETA_SFT'] and not SFT_MERGED.exists():
    from transformers import AutoModelForCausalLM, AutoTokenizer

    print('Creating theta_SFT snapshot from TinyLlama-1.1B-Chat:', SFT_MERGED)
    SFT_MERGED.mkdir(parents=True, exist_ok=True)
    snapshot_tokenizer = AutoTokenizer.from_pretrained(CONFIG['BASE_MODEL'])
    snapshot_model = AutoModelForCausalLM.from_pretrained(
        CONFIG['BASE_MODEL'],
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    )
    snapshot_model.save_pretrained(str(SFT_MERGED), safe_serialization=True)
    snapshot_tokenizer.save_pretrained(str(SFT_MERGED))
    del snapshot_model, snapshot_tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
elif SFT_MERGED.exists():
    print('theta_SFT snapshot available:', SFT_MERGED)
else:
    print('USE_BASE_AS_THETA_SFT is False; run the SFT phase before Head-Sanity/PPO.')

if CONFIG['HEAD_SANITY_ON_GENERATIONS']:
    assert SFT_MERGED.exists(), (
        'theta_SFT missing. With USE_BASE_AS_THETA_SFT=True this cell creates it; '
        'otherwise run the SFT phase first with the same OUTPUT_DIR.')


In [ ]:
HEAD_SANITY_PATH = OUTPUT_DIR / 'head_sanity.json'

if CONFIG['RUN_HEAD_SANITY']:
    from datasets import load_dataset
    from scipy.stats import pearsonr

    # ---- 1) VRAM -----------------------------------------------------------
    assert torch.cuda.is_available(), 'ArmoRM PPO path needs CUDA.'
    total_gb = torch.cuda.mem_get_info()[1] / 1024**3
    print(f'GPU: {torch.cuda.get_device_name(0)}  total={total_gb:.1f} GB')
    assert total_gb >= CONFIG['MIN_GPU_MEMORY_GB'], (
        f'A100-40GB-class runtime expected; got {total_gb:.1f} GB. '
        f'ArmoRM(4bit) + policy + reference model + value head will not fit.')

    # ---- 2) F2: load scorer; index resolved BY NAME, asserted --------------
    scorer = rs_ppo.ArmoRMHeadScorer(axis='helpfulness', model_id=CONFIG['ARMORM_MODEL'])
    hs_indices = scorer.helpsteer_indices()
    resolved = {a: int(i) for a, i in zip(CONFIG['ATTRIBUTES'], hs_indices)}
    print('Resolved ArmoRM head indices (by name, not assumed):', resolved)
    assert len(set(hs_indices)) == 5, 'duplicate head indices resolved'

    # ---- 3) F3: prove batched == single ------------------------------------
    ds = load_dataset('nvidia/HelpSteer2', split=CONFIG['HEAD_SANITY_SPLIT'])
    rng = np.random.default_rng(CONFIG['SEED'])
    idx = rng.choice(len(ds), size=CONFIG['HEAD_SANITY_SAMPLES'], replace=False)
    rows = [ds[int(i)] for i in idx]
    prompts = [str(r['prompt']) for r in rows]
    labels = np.asarray([[float(r[a]) for a in CONFIG['ATTRIBUTES']] for r in rows],
                        dtype=np.float64)

    probe_pairs = [(p, 'This is a short probe response used to validate the scorer.')
                   for p in prompts[:8]]
    batching_report = scorer.validate_batching(probe_pairs)
    print('Batching report:', json.dumps(batching_report, indent=2))

    # ---- 4) F5: score the distribution PPO will actually produce -----------
    if CONFIG['HEAD_SANITY_ON_GENERATIONS']:
        assert SFT_MERGED.exists(), 'theta_SFT snapshot missing; run the theta_SFT snapshot cell first.'
        responses = rs_ppo.generate_responses(str(SFT_MERGED), prompts, seed=CONFIG['SEED'])
        distribution = 'theta_SFT generations (16-32 tokens, PPO gen-config)'
    else:
        responses = [str(r['response']) for r in rows]
        distribution = 'HelpSteer2 reference responses (WRONG distribution - diagnostic only)'
    print('Sanity distribution:', distribution)

    scores = scorer.score_all_heads(prompts, responses)
    assert scores.shape == (CONFIG['HEAD_SANITY_SAMPLES'], 5)
    assert np.all(np.isfinite(scores)), 'non-finite ArmoRM scores'

    # ---- degeneracy assertions (the QRM lesson) ----------------------------
    stds = scores.std(axis=0)
    uniques = [int(len(np.unique(scores[:, i]))) for i in range(5)]
    failures = []
    for i, axis in enumerate(CONFIG['ATTRIBUTES']):
        if stds[i] <= CONFIG['HEAD_STD_MIN']:
            failures.append(f'{axis}: constant head (std={stds[i]:.3e})')
        if uniques[i] <= CONFIG['HEAD_UNIQUE_MIN']:
            failures.append(f'{axis}: only {uniques[i]} unique values')

    # ---- axis-discriminativeness (diag of ArmoRM x HelpSteer2 labels) ------
    cross = np.full((5, 5), np.nan)
    for i in range(5):
        for j in range(5):
            if scores[:, i].std() > 0 and labels[:, j].std() > 0:
                cross[i, j] = float(pearsonr(scores[:, i], labels[:, j]).statistic)
    diag = np.diag(cross)
    row_argmax_on_diag = [bool(np.nanargmax(cross[i]) == i) for i in range(5)]
    n_discriminative = int(sum(row_argmax_on_diag))
    print(f'Axis-discriminative heads (row argmax on own axis): {n_discriminative}/5')

    passed = len(failures) == 0
    head_sanity = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'n': int(scores.shape[0]),
        'distribution': distribution,
        'on_generations': bool(CONFIG['HEAD_SANITY_ON_GENERATIONS']),
        'attributes': CONFIG['ATTRIBUTES'],
        'resolved_head_indices': resolved,
        'batching_report': batching_report,
        'std': dict(zip(CONFIG['ATTRIBUTES'], stds.tolist())),
        'unique_counts': dict(zip(CONFIG['ATTRIBUTES'], uniques)),
        'armorm_vs_helpsteer2_pearson': cross.tolist(),
        'diagonal': dict(zip(CONFIG['ATTRIBUTES'], diag.tolist())),
        'n_axis_discriminative': n_discriminative,
        'failures': failures,
        'passed': passed,
        'circularity_acknowledged': True,
    }
    write_json(HEAD_SANITY_PATH, head_sanity)
    write_numpy(OUTPUT_DIR / 'head_sanity_scores.npy', scores)
    write_numpy(OUTPUT_DIR / 'head_sanity_labels.npy', labels)
    display(pd.DataFrame(cross, index=CONFIG['ATTRIBUTES'], columns=CONFIG['ATTRIBUTES']))

    del scorer
    torch.cuda.empty_cache()

    assert passed, 'HEAD SANITY FAILED -> DO NOT TRAIN:\n  ' + '\n  '.join(failures)
    print('Head sanity PASSED. Safe to train.')
    if n_discriminative < 5:
        print(f'NOTE: only {n_discriminative}/5 heads are axis-discriminative. '
              f'Not a blocker, but report this honestly (v5 found 2/5 self-maximizing).')
else:
    write_json(HEAD_SANITY_PATH, {'passed': False, 'skipped': True,
                                  'reason': 'RUN_HEAD_SANITY is False'})
    print('Head sanity skipped.')


## Phase 1 and 2 - SFT Base and the Five PPO Runs

**F1:** Both phases run **in process** via `rs_ppo.run_sft()` / `rs_ppo.run_ppo(...)`. Therefore `out_dir` (-> `OUTPUT_DIR/rs_runs`), `batch_size`, `total_ppo_steps`, `n_prompts`, and 4-bit settings really take effect.

**Equal-N:** identical `prompt_seed=137`, `train_seed=911`, identical step count for **all** axes. No per-axis tuning (supervisor guardrail: "training is secondary").

**F6:** After each run, `reward_plateaued` is checked. Plateau = converged = Assumption 1 broken = upper-bound argument gone.

`RUN_SFT` is only needed when `USE_BASE_AS_THETA_SFT=False`. With `USE_BASE_AS_THETA_SFT=True`, the shared init snapshot is already TinyLlama-1.1B-Chat.


In [ ]:
PLATEAU_PATH = OUTPUT_DIR / 'plateau_report.json'

if CONFIG['RUN_SFT']:
    print('=== Phase 1: SFT -> theta_SFT (shared init for all PPO runs) ===')
    sft_path = rs_ppo.run_sft()
    print('theta_SFT:', sft_path)
else:
    print('RUN_SFT is False; SFT phase not started. This is expected when USE_BASE_AS_THETA_SFT=True.')
    if SFT_MERGED.exists():
        print('  (existing theta_SFT found at', SFT_MERGED, ')')

if CONFIG['RUN_PPO']:
    head_sanity = json.loads(HEAD_SANITY_PATH.read_text(encoding='utf-8'))
    assert head_sanity.get('passed') is True, 'Head sanity did not pass; DO NOT TRAIN.'
    assert SFT_MERGED.exists(), 'theta_SFT missing; run the SFT phase first.'

    plateau_report = {}
    for axis in CONFIG['PPO_AXES']:
        print(f'\n=== Phase 2: PPO run for axis {axis!r} (CIRCULAR: reward = ArmoRM) ===')
        result = rs_ppo.run_ppo(
            axis,
            reward_model_id=CONFIG['ARMORM_MODEL'],
            circular_armorm_acknowledged=CONFIG['CIRCULAR_ARMORM_ACKNOWLEDGED'],
        )
        plateau_report[axis] = result['plateau']
        adapter_dir = RS_RUNS_DIR / f'ppo_{axis}' / 'adapter'
        assert adapter_dir.is_dir(), f'adapter not written to expected path: {adapter_dir}'
        print(f'  -> adapter at {adapter_dir}')

    plateaued = [a for a, p in plateau_report.items() if p.get('reward_plateaued') is True]
    plateau_report['_summary'] = {
        'axes_plateaued': plateaued,
        'upper_bound_reading_valid': len(plateaued) == 0,
        'note': ('The upper-bound claim requires a SHORT horizon on every axis. Any axis that '
                 'plateaued has converged -> delta != eta*grad r -> Assumption 1 broken there.'),
    }
    write_json(PLATEAU_PATH, plateau_report)
    if plateaued:
        print(f'\nWARNING: axes plateaued: {plateaued}. The best-case/upper-bound reading '
              f'is NOT valid for these axes. Report this, do not hide it.')
    else:
        print('\nNo axis plateaued -> short-horizon regime intact; upper-bound reading holds.')
else:
    write_json(PLATEAU_PATH, {'pending': True, 'reason': 'RUN_PPO is False'})
    print('RUN_PPO is False. Planned in-process calls:')
    for axis in CONFIG['PPO_AXES']:
        print(f"  rs_ppo.run_ppo({axis!r}, reward_model_id={CONFIG['ARMORM_MODEL']!r}, "
              f"circular_armorm_acknowledged=True)")


## Phase 3 - Geometry: Delta Norms, R, R-Minus, Floor LP

**Not circular** (0 reward queries). The central question: **does PPO break the conflict-free geometry?** On HelpSteer2/SFT, `R_cos` was entirely positive off-diagonal (0.299-0.397) -> `R- = 0` -> floor collapse `F_p = {p}` -> C1++/M1++/P2++/P3++ all return `p`.

**F7:** Floor collapse is now decided **from the LP** (`max_{1^T v = 0, ||v||_1 <= 1} min_i (Rv)_i <= tol`), not from the sign of the off-diagonals. The LP *is* the criterion; `R- != 0` was only a heuristic for it.


In [ ]:
DELTA_NORMS_PATH = OUTPUT_DIR / 'delta_norms.json'
GEOMETRY_PRECHECK_PATH = OUTPUT_DIR / 'geometry_precheck.json'
R_GRAM_PATH = OUTPUT_DIR / 'R_gram.npy'
R_COS_PATH = OUTPUT_DIR / 'R_cos.npy'


from src.coefficient_portfolio import floor_lp

if CONFIG['RUN_GEOMETRY']:
    from src.effective_lora_geometry import (effective_lora_inner_product,
                                             load_effective_lora_geometry,
                                             validate_compatible_geometries)
    adapter_paths = {a: RS_RUNS_DIR / f'ppo_{a}' / 'adapter' for a in CONFIG['ATTRIBUTES']}
    missing = [str(p) for p in adapter_paths.values() if not p.is_dir()]
    assert not missing, 'Missing PPO adapters: ' + ', '.join(missing)

    geometries = {a: load_effective_lora_geometry(p) for a, p in adapter_paths.items()}
    validate_compatible_geometries([geometries[a] for a in CONFIG['ATTRIBUTES']],
                                   CONFIG['ATTRIBUTES'])

    n = len(CONFIG['ATTRIBUTES'])
    gram = np.zeros((n, n))
    for i, left in enumerate(CONFIG['ATTRIBUTES']):
        for j, right in enumerate(CONFIG['ATTRIBUTES']):
            gram[i, j] = effective_lora_inner_product(geometries[left], geometries[right])
    gram = 0.5 * (gram + gram.T)
    norms = np.sqrt(np.maximum(np.diag(gram), 0.0))
    assert np.all(norms > 0), 'a PPO adapter has a zero task vector (no learning happened)'
    cos = gram / np.outer(norms, norms)
    np.fill_diagonal(cos, 1.0)

    offdiag = cos[~np.eye(n, dtype=bool)]
    neg = offdiag[offdiag < 0]

    # F7: the LP IS the floor criterion
    lp_value, floor_collapsed = floor_lp(cos, CONFIG['FLOOR_TOL'])

    write_numpy(R_GRAM_PATH, gram)
    write_numpy(R_COS_PATH, cos)
    write_json(DELTA_NORMS_PATH, {
        'attributes': CONFIG['ATTRIBUTES'],
        'delta_norm': dict(zip(CONFIG['ATTRIBUTES'], norms.tolist())),
        'mean_norm': float(norms.mean()),
        'max_percent_deviation_from_mean': float(np.max(np.abs(norms / norms.mean() - 1.0)) * 100),
        'note': 'Equal-N should keep these uniform (SFT reference: 5.33-5.46).',
    })
    write_json(GEOMETRY_PRECHECK_PATH, {
        'attributes': CONFIG['ATTRIBUTES'],
        'R_minus_nonzero': bool(len(neg) > 0),
        'negative_offdiag_count': int(len(neg)),
        'negative_offdiag_min': float(neg.min()) if len(neg) else None,
        'cosine_offdiag_min': float(offdiag.min()),
        'cosine_offdiag_max': float(offdiag.max()),
        'floor_lp_max_min_Rv': lp_value,
        'floor_lp_l1_bound': 1.0,
        'floor_collapsed': floor_collapsed,       # F7: THE criterion
        'floor_tol': CONFIG['FLOOR_TOL'],
        'interpretation': ('floor_collapsed=True  -> F_p = {p}: C1++/M1++/P2++/P3++ all return p '
                           '(report as certificates, not failures); only M1+ can move. '
                           'floor_collapsed=False -> PPO broke the conflict-free geometry; the '
                           'vector-safe portfolio becomes non-trivial for the first time.'),
        'circular_do_not_report_as_proxy_validation': True,
    })
    display(pd.DataFrame(cos, index=CONFIG['ATTRIBUTES'], columns=CONFIG['ATTRIBUTES']))
    print(f'\nR-minus nonzero: {len(neg) > 0}  (negative off-diagonals: {len(neg)})')
    print(f'Floor LP value  : {lp_value:.6e}')
    print(f'FLOOR COLLAPSED : {floor_collapsed}  -> ' +
          ('F_p = {p}; vector-safe methods return p.' if floor_collapsed
           else 'floor is OPEN; vector-safe methods can move.'))
else:
    write_json(DELTA_NORMS_PATH, {'pending': True, 'reason': 'RUN_GEOMETRY is False'})
    write_json(GEOMETRY_PRECHECK_PATH, {'pending': True, 'reason': 'RUN_GEOMETRY is False'})
    print('RUN_GEOMETRY is False; geometry skipped.')


## Phase 4a - Reward Collection over the Search Set B (F8)

The most expensive step, and the prerequisite for **everything** that follows: for each `lambda` in B, `theta(lambda)` is **actually merged**, generations are produced over 80 held-out prompts, and all five ArmoRM heads are scored. Result: `reward_matrix.npy` with shape `(|B|, 5)`.

Prompts: `validation` slice starting at offset **160** - disjoint from the v5 confirmatory slice `[80:160]`.

Rows 0-4 of B are the vertices (= the five PPO specialists) and provide the STL reference for `Delta m%`.


In [ ]:
from src.proxy_validation import build_search_set
from src.merge import merge_delta_norm_check, merge_theta
from src.preferences import CV_PREFS, PREFERENCES, QUALITY_PREFS

SEARCH_SET_PATH = OUTPUT_DIR / 'search_set_B.npy'
REWARD_MATRIX_PATH = OUTPUT_DIR / 'reward_matrix.npy'
REWARD_MATRIX_META_PATH = OUTPUT_DIR / 'reward_matrix_meta.json'
REWARD_PROMPTS_PATH = OUTPUT_DIR / 'reward_prompts.json'

B = build_search_set(5, n_dirichlet=CONFIG['SEARCH_SET_DIRICHLET'],
                     preferences=list(PREFERENCES.values()),
                     seed=CONFIG['SEARCH_SET_SEED'])
write_numpy(SEARCH_SET_PATH, B)
print('Search set B:', B.shape, '(rows 0-4 = vertices = the five PPO specialists)')
assert np.allclose(B[:5], np.eye(5)), 'rows 0-4 of B must be the vertices'


if CONFIG['RUN_REWARD_COLLECTION']:
    from datasets import load_dataset
    from transformers import AutoTokenizer

    adapter_paths = {a: RS_RUNS_DIR / f'ppo_{a}' / 'adapter' for a in CONFIG['ATTRIBUTES']}
    for p in adapter_paths.values():
        assert p.is_dir(), f'missing adapter: {p}'
    assert SFT_MERGED.exists(), 'theta_SFT missing'

    merge_rel_error = merge_delta_norm_check(B[5], adapter_paths)
    assert merge_rel_error < 1e-6, f'effective merge linearity check failed: {merge_rel_error}'
    try:
        git_sha = subprocess.check_output(
            ['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
    except Exception:
        git_sha = 'unknown'

    ds = load_dataset('nvidia/HelpSteer2', split=CONFIG['REWARD_PROMPT_SPLIT'])
    off = CONFIG['REWARD_PROMPT_OFFSET']
    eval_prompts = [str(ds[i]['prompt'])
                    for i in range(off, off + CONFIG['REWARD_NUM_PROMPTS'])]
    write_json(REWARD_PROMPTS_PATH, {'split': CONFIG['REWARD_PROMPT_SPLIT'],
                                     'offset': off, 'n': len(eval_prompts),
                                     'disjoint_from_v5_confirmatory_slice': '[80:160]',
                                     'prompts': eval_prompts})

    scorer = rs_ppo.ArmoRMHeadScorer(axis='helpfulness', model_id=CONFIG['ARMORM_MODEL'])
    scorer.validate_batching([(p, 'probe response') for p in eval_prompts[:8]])

    tok = AutoTokenizer.from_pretrained(str(SFT_MERGED))
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'left'

    length_rng = np.random.default_rng(CONFIG['SEED'])
    generation_lengths = [int(length_rng.integers(rs_ppo.CFG['output_min_len'],
                                                  rs_ppo.CFG['output_max_len'] + 1))
                          for _ in range(CONFIG['N_GEN_PER_PROMPT'])]

    reward_rows = []
    t0 = time.time()
    for k, lmbda in enumerate(B):
        rs_ppo.set_all_seeds(CONFIG['SEED'])          # same rollouts for every lambda
        model = merge_theta(lmbda, adapter_paths, SFT_MERGED)
        repeat_heads = []
        for repeat, n_new in enumerate(generation_lengths):
            responses = []
            for start in range(0, len(eval_prompts), 8):
                chunk = eval_prompts[start:start + 8]
                texts = [tok.apply_chat_template([{'role': 'user', 'content': p}],
                                                 tokenize=False, add_generation_prompt=True)
                         for p in chunk]
                enc = tok(texts, return_tensors='pt', padding=True).to(model.device)
                with torch.inference_mode():
                    out = model.generate(**enc, max_new_tokens=n_new, do_sample=True,
                                         top_k=0, top_p=1.0, pad_token_id=tok.eos_token_id)
                responses.extend(tok.batch_decode(out[:, enc['input_ids'].shape[1]:],
                                                  skip_special_tokens=True))
            repeat_heads.append(scorer.score_all_heads(eval_prompts, responses))
        heads = np.mean(np.stack(repeat_heads, axis=0), axis=0)   # [80, 5]
        reward_rows.append(heads.mean(axis=0))
        del model
        torch.cuda.empty_cache()
        if (k + 1) % 5 == 0 or k == len(B) - 1:
            el = time.time() - t0
            print(f'  lambda {k+1}/{len(B)}  elapsed={el/60:.1f} min  '
                  f'eta={(el/(k+1)*(len(B)-k-1))/60:.1f} min')

    Reward = np.asarray(reward_rows, dtype=np.float64)
    assert Reward.shape == (B.shape[0], 5)
    assert np.all(np.isfinite(Reward))
    # cache-contamination guard (v4 lesson): vertex rows must differ from one another
    assert len(np.unique(np.round(Reward[:5], 8), axis=0)) == 5, \
        'vertex reward rows are identical -> cache contamination or merge is a no-op'
    write_numpy(REWARD_MATRIX_PATH, Reward)
    write_json(REWARD_MATRIX_META_PATH, {
        'merge_impl': 'src.merge.merge_theta',
        'merge_linearity_verified': True,
        'merge_delta_norm_rel_error': float(merge_rel_error),
        'git_sha': git_sha,
        'n_gen_per_prompt': CONFIG['N_GEN_PER_PROMPT'],
        'generation_lengths': generation_lengths,
        'prompt_split': CONFIG['REWARD_PROMPT_SPLIT'],
        'prompt_offset': CONFIG['REWARD_PROMPT_OFFSET'],
        'n_prompts': CONFIG['REWARD_NUM_PROMPTS'],
        'reward_matrix_path': str(REWARD_MATRIX_PATH),
    })
    print('reward_matrix.npy written:', Reward.shape)
    print('reward_matrix_meta.json written:', REWARD_MATRIX_META_PATH)
    display(pd.DataFrame(Reward[:5], index=CONFIG['ATTRIBUTES'], columns=CONFIG['ATTRIBUTES']))
    del scorer
    torch.cuda.empty_cache()
else:
    print('RUN_REWARD_COLLECTION is False; no reward matrix built (no fake numbers).')


## Phase 4b - Wall-A Test (R2) and LMC

**Wall A** is the actual go/no-go check. Linear vertex fit per axis against the *true* vertex rewards, `R2` on the non-vertex points of B. If `quality R2 ~ 0` or negative, **no** endpoint-linear rule can help there; neither `R` nor a measured `H` can resolve it. This holds regardless of how good the training was, and is **not circular**: nonlinearity is a property of ArmoRM x interpolation.

**LMC** (RS Working Hyp. 1): reward along `theta_SFT <-> theta_i`. RS itself warns that fully antagonistic rewards can break linear mode connectivity - the basic precondition of the entire interpolation family.


In [ ]:
LINEARITY_R2_PATH = OUTPUT_DIR / 'linearity_r2.json'
LMC_CHECK_PATH = OUTPUT_DIR / 'lmc_check.json'

# ---- Wall A -----------------------------------------------------------------
if REWARD_MATRIX_PATH.exists():
    assert REWARD_MATRIX_META_PATH.exists(), 'reward_matrix_meta.json missing; old reward matrix is not trusted'
    reward_meta = json.loads(REWARD_MATRIX_META_PATH.read_text(encoding='utf-8'))
    assert reward_meta.get('merge_impl') == 'src.merge.merge_theta', reward_meta
    assert reward_meta.get('merge_linearity_verified') is True, reward_meta
    Reward = np.load(REWARD_MATRIX_PATH)
    assert Reward.shape == (B.shape[0], 5)
    vertex_rewards = Reward[:5]                      # [vertex k, axis j]
    mask = np.arange(len(B)) >= 5                    # evaluate off the vertices

    rows = []
    for j, axis in enumerate(CONFIG['ATTRIBUTES']):
        y = Reward[:, j]
        y_hat = B @ vertex_rewards[:, j]             # endpoint-linear prediction
        ss_res = float(np.sum((y[mask] - y_hat[mask]) ** 2))
        ss_tot = float(np.sum((y[mask] - y[mask].mean()) ** 2))
        r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else None
        rows.append({'axis': axis, 'r2_vertex_linear_fit': r2,
                     'regime': 'quality' if axis in ('helpfulness', 'correctness', 'coherence')
                               else 'complexity/verbosity'})
    quality_r2 = [r['r2_vertex_linear_fit'] for r in rows if r['regime'] == 'quality'
                  and r['r2_vertex_linear_fit'] is not None]
    wall_a_stands = bool(quality_r2 and max(quality_r2) < CONFIG['WALL_A_R2_THRESHOLD'])

    write_json(LINEARITY_R2_PATH, {
        'circular_do_not_report_as_proxy_validation': True,
        'rows': rows,
        'quality_r2_max': float(max(quality_r2)) if quality_r2 else None,
        'wall_A_R2_threshold': CONFIG['WALL_A_R2_THRESHOLD'],
        'wall_a_stands': wall_a_stands,
        'merge_provenance': reward_meta,
        'interpretation': ('wall_a_stands=True: U_p is nonlinear in lambda on the quality axes. '
                           'No endpoint-linear rule (R-based or H-based) can trace it -- even in '
                           'this best-case circular regime. This is the upper-bound negative '
                           'result and it is a substantive thesis result. '
                           'wall_a_stands=False: Wall A was not a hard cap; strong positive.'),
        'v5_reference': 'SFT/HelpSteer2: quality R2 ~ 0 or negative; compl/verb up to 0.77',
    })
    display(pd.DataFrame(rows))
    print(f'\nWALL A STANDS: {wall_a_stands}')
else:
    write_json(LINEARITY_R2_PATH, {'pending': True,
                                   'reason': 'reward_matrix.npy missing; run RUN_REWARD_COLLECTION'})
    print('Wall A pending: no reward_matrix.npy.')

# ---- LMC --------------------------------------------------------------------
if CONFIG['RUN_LMC']:
    from datasets import load_dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer

    adapter_paths = {a: RS_RUNS_DIR / f'ppo_{a}' / 'adapter' for a in CONFIG['ATTRIBUTES']}
    ds = load_dataset('nvidia/HelpSteer2', split=CONFIG['REWARD_PROMPT_SPLIT'])
    off = CONFIG['REWARD_PROMPT_OFFSET']
    eval_prompts = [str(ds[i]['prompt']) for i in range(off, off + 32)]   # LMC: 32 prompts suffice

    scorer = rs_ppo.ArmoRMHeadScorer(axis='helpfulness', model_id=CONFIG['ARMORM_MODEL'])
    scorer.validate_batching([(p, 'probe response') for p in eval_prompts[:8]])
    tok = AutoTokenizer.from_pretrained(str(SFT_MERGED))
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'left'

    curves = {}
    for i, axis in enumerate(CONFIG['ATTRIBUTES']):
        curve = []
        for t in CONFIG['LMC_GRID']:
            lmbda = np.zeros(5); lmbda[i] = t       # theta_SFT (t=0) -> theta_i (t=1)
            rs_ppo.set_all_seeds(CONFIG['SEED'])
            if np.isclose(t, 0.0):
                model = AutoModelForCausalLM.from_pretrained(
                    str(SFT_MERGED), torch_dtype=torch.bfloat16, device_map='auto')
            else:
                model = merge_theta(lmbda, adapter_paths, SFT_MERGED)
            responses = []
            for start in range(0, len(eval_prompts), 8):
                chunk = eval_prompts[start:start + 8]
                texts = [tok.apply_chat_template([{'role': 'user', 'content': p}],
                                                 tokenize=False, add_generation_prompt=True)
                         for p in chunk]
                enc = tok(texts, return_tensors='pt', padding=True).to(model.device)
                with torch.inference_mode():
                    out = model.generate(**enc, max_new_tokens=24, do_sample=True,
                                         top_k=0, top_p=1.0, pad_token_id=tok.eos_token_id)
                responses.extend(tok.batch_decode(out[:, enc['input_ids'].shape[1]:],
                                                  skip_special_tokens=True))
            heads = scorer.score_all_heads(eval_prompts, responses)
            curve.append(float(heads.mean(axis=0)[i]))   # own-axis reward
            del model; torch.cuda.empty_cache()
        curves[axis] = curve
        print(f'[lmc] {axis}: {[round(c, 4) for c in curve]}')

    # RS Working Hyp. 1: reward along the path >= linear interpolation of endpoints
    lmc = {}
    for axis, curve in curves.items():
        lo, hi = curve[0], curve[-1]
        lin = [lo + t * (hi - lo) for t in CONFIG['LMC_GRID']]
        deficits = [c - l for c, l in zip(curve, lin)]
        lmc[axis] = {'grid': CONFIG['LMC_GRID'], 'reward': curve,
                     'linear_interp': lin, 'deficit': deficits,
                     'min_deficit': float(min(deficits)),
                     'lmc_holds': bool(min(deficits) >= -0.01)}
    lmc['_summary'] = {
        'lmc_holds_all_axes': all(v['lmc_holds'] for k, v in lmc.items() if not k.startswith('_')),
        'note': ('RS Appendix B.3.2 warns that fully antagonistic rewards can break linear mode '
                 'connectivity -- and LMC is the precondition of the whole interpolation family.'),
    }
    write_json(LMC_CHECK_PATH, lmc)
    print('\nLMC holds on all axes:', lmc['_summary']['lmc_holds_all_axes'])
    del scorer; torch.cuda.empty_cache()
else:
    write_json(LMC_CHECK_PATH, {'pending': True, 'reason': 'RUN_LMC is False'})
    print('RUN_LMC is False; LMC skipped.')


## Phase 5 - The Binding Test: M1+ vs. lambda=p with Real Merges (F8)

The **primary preregistered endpoint**. For each preference `p`:

1. `lambda* = M1+(p, R_cos)` via SLSQP (`rho=0.5`).
2. **Actually merge** `theta(lambda*)` and `theta(p)`. No nearest-point-in-B approximation.
3. Primary `Delta U_p = U_p(lambda*) - U_p(p)` is measured as a **paired per-prompt delta on the raw ArmoRM scale**; rank-normalized `U_p` is secondary.
4. **Bootstrap 95% CI** over the 80 prompts (2000 resamples). An effect only counts if the CI excludes 0.
5. `Delta m%` gain (`lambda*` minus `p`) against the specialist reference.

**All five methods run** - M1+, C1++ (with trust region, as in the portfolio definition), M1++, P2++, P3++. If the four vector-safe methods return `p`, this is the **floor certificate** ("no direction improves all objectives simultaneously") and is reported as a theorem, not as a failure.

**Metric correction:** The primary endpoint is the **paired per-prompt delta** on the raw ArmoRM scale - only this can be bootstrapped over the 80 prompts. Rank-normalized `U_p` is a **set-level** quantity (ranks over B) and cannot be resampled prompt-wise; it runs as a **secondary** metric, together with a paired rank transform (ranks pooled over both models) that is bootstrappable. Both are fixed **before** the binding run.

**Negative control:** the quality-heavy preferences must run.


## Phase 5 - Final Merge Test Moved to Notebook 09

The binding primary endpoint is no longer computed in this notebook. Run `notebooks/09_final_merge_test_colab.ipynb` after Notebook 08 has produced `R_cos.npy`, `reward_matrix.npy`, `reward_matrix_meta.json`, `search_set_B.npy`, the PPO adapters, and `theta_sft/merged`.

Notebook 09 performs the real `lambda*` merges with `src.merge.merge_theta`, Holm-corrected preference-family decisions, and the final `merge_results.json`.


## Verdict, Summary, and Zip

The verdict remains **STOP** as long as required artifacts are missing. In addition to the original design, the **plateau status** is tracked explicitly: if an axis plateaus, the upper-bound interpretation is invalid for that axis - this must appear in the verdict, not in a footnote.


In [ ]:
VERDICT_PATH = OUTPUT_DIR / 'verdict.json'
SUMMARY_PATH = OUTPUT_DIR / 'summary.md'
ZIP_PATH = OUTPUT_DIR / CONFIG['OUTPUT_ZIP']

artifacts = {
    'preregistration': PREREGISTRATION_PATH,
    'head_sanity': HEAD_SANITY_PATH,
    'plateau_report': PLATEAU_PATH,
    'delta_norms': DELTA_NORMS_PATH,
    'R_gram': R_GRAM_PATH,
    'R_cos': R_COS_PATH,
    'geometry_precheck': GEOMETRY_PRECHECK_PATH,
    'lmc_check': LMC_CHECK_PATH,
    'reward_matrix': REWARD_MATRIX_PATH,
    'reward_matrix_meta': REWARD_MATRIX_META_PATH,
    'reward_prompts': REWARD_PROMPTS_PATH,
    'search_set_B': SEARCH_SET_PATH,
    'linearity_r2': LINEARITY_R2_PATH,
}
loaded = {k: json.loads(Path(p).read_text(encoding='utf-8'))
          for k, p in artifacts.items() if Path(p).exists() and Path(p).suffix == '.json'}

reasons = []
missing = [k for k, p in artifacts.items() if not Path(p).exists()]
if missing:
    reasons.append('missing outputs: ' + ', '.join(missing))
if loaded.get('head_sanity', {}).get('passed') is not True:
    reasons.append('head sanity not passed')

plateau = loaded.get('plateau_report', {})
if plateau.get('pending'):
    reasons.append('plateau check pending')
elif plateau.get('_summary', {}).get('upper_bound_reading_valid') is False:
    reasons.append('reward plateaued on ' + ', '.join(plateau['_summary']['axes_plateaued'])
                   + ' -> upper-bound reading INVALID on those axes')

geom = loaded.get('geometry_precheck', {})
if geom.get('pending'):
    reasons.append('geometry pending')
elif geom.get('floor_collapsed') is True:
    reasons.append('floor collapsed (F_p = {p}) -> vector-safe methods return p; only M1+ moves')

lin = loaded.get('linearity_r2', {})
if lin.get('pending'):
    reasons.append('Wall-A / linearity R2 pending')

verdict = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'circularity_acknowledged': True,
    'retired_rqs': ['RQ2 proxy validity'],
    'valid_claims': ['upper bound', 'R-minus', 'wall A (R2)', 'LMC'],
    'invalid_claims': ['proxy validity', 'generalization beyond this reward model'],
    'circular_do_not_report_as_proxy_validation': True,
    'upper_bound_reading_valid': plateau.get('_summary', {}).get('upper_bound_reading_valid'),
    'floor_collapsed': geom.get('floor_collapsed'),
    'wall_a_stands': lin.get('wall_a_stands'),
    'primary_endpoint_location': 'notebooks/09_final_merge_test_colab.ipynb',
    'primary_success': None,
    'upper_bound_verdict': None,
    'decision': 'GO' if not reasons else 'STOP',
    'reasons': reasons,
    'config': CONFIG,
}
write_json(VERDICT_PATH, verdict)

summary = ['# 08 - RS-PPO against ArmoRM (deliberately circular)', '']
summary += ['## What this notebook can and cannot show', '',
            '**Valid:** upper bound (best-case regime), R-minus, Wall A / R2, LMC.',
            '',
            '**Invalid:** RQ2 proxy validity and any generalization beyond ArmoRM.',
            '',
            'PPO is deliberately trained against ArmoRM and evaluated against ArmoRM. This is circular '
            'and explicitly acknowledged in `preregistration.json`. Spearman/gate numbers from this '
            'run must **not** be reported as proxy validation.', '',
            '**Condition for the upper bound:** short horizon. If the reward curve plateaus, '
            '`delta ~ eta*grad r` is violated and the best-case interpretation fails.', '',
            f"## Verdict: **{verdict['decision']}**", '']
summary += ['Reasons:'] + (['- ' + r for r in reasons] if reasons
                           else ['- all preregistered criteria satisfied'])
summary += ['', '## Key Findings', '',
            f"- upper_bound_reading_valid: {verdict['upper_bound_reading_valid']}",
            f"- floor_collapsed: {verdict['floor_collapsed']}",
            f"- wall_a_stands: {verdict['wall_a_stands']}",
            '- primary endpoint: moved to notebooks/09_final_merge_test_colab.ipynb',
            '']
summary += ['## Outputs', ''] + [f'- {k}: {Path(p).name}' for k, p in artifacts.items()]
SUMMARY_PATH.write_text('\n'.join(summary) + '\n', encoding='utf-8')

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for p in [*artifacts.values(), VERDICT_PATH, SUMMARY_PATH]:
        p = Path(p)
        if p.exists():
            archive.write(p, arcname=p.name)
            print('Added:', p.name)

print('\nOutput zip:', ZIP_PATH)
print('Verdict   :', verdict['decision'])
for r in reasons:
    print('  -', r)

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except Exception as error:
    print('Download manually from:', ZIP_PATH, '|', repr(error))
